In [4]:
import os
os.environ["MILK10K_DIR"] = "/Users/yourname/Desktop/MILK10k-Lesion-Classification/MILK10k-Lesion-Classification/data"

In [13]:
import os
DATA_DIR = os.environ.get("MILK10K_DIR", "../data/milk10k")

In [14]:
import subprocess
result = subprocess.run(["find", "../data", "-iname", "*gt*"], capture_output=True, text=True)
print(result.stdout)

../data/milk10k/supplements/training_gt.csv



In [17]:
import os

DATA_DIR = "../data/milk10k"
GT_PATH = os.path.join(DATA_DIR, "supplements", "training_gt.csv")
METADATA_PATH = os.path.join(DATA_DIR, "metadata.csv")

metadata = pd.read_csv(METADATA_PATH)
gt = pd.read_csv(GT_PATH)

print(metadata.shape, gt.shape)

(10480, 17) (5240, 12)


A1. SESSION 1 - Trusting and shaping the data (pandas)

A1.1 Cross-check the two label files.

In [18]:
import pandas as pd
import os

# --- Config: single source of truth for the data location ---
DATA_DIR = "../data/milk10k"
METADATA_PATH = os.path.join(DATA_DIR, "metadata.csv")
GT_PATH = os.path.join(DATA_DIR, "supplements", "training_gt.csv")  # nested under supplements/

metadata = pd.read_csv(METADATA_PATH)
gt = pd.read_csv(GT_PATH)

print("metadata shape:", metadata.shape)
print("gt shape:", gt.shape)


# ============================================================
# (a) lesion_id sets identical in both files
# ============================================================
meta_ids = set(metadata["lesion_id"])
gt_ids = set(gt["lesion_id"])

only_in_meta = meta_ids - gt_ids
only_in_gt = gt_ids - meta_ids

print("\n(a) lesion_ids only in metadata:", len(only_in_meta))
print("(a) lesion_ids only in gt:      ", len(only_in_gt))

assert only_in_meta == set(), f"lesion_ids missing from gt: {only_in_meta}"
assert only_in_gt == set(), f"lesion_ids missing from metadata: {only_in_gt}"
assert meta_ids == gt_ids
print("(a) PASSED: lesion_id sets are identical")

metadata shape: (10480, 17)
gt shape: (5240, 12)

(a) lesion_ids only in metadata: 0
(a) lesion_ids only in gt:       0
(a) PASSED: lesion_id sets are identical


In [19]:
# ============================================================
# (b) exactly one positive class per lesion in the one-hot gt
# ============================================================
class_cols = [c for c in gt.columns if c != "lesion_id"]
row_sums = gt[class_cols].sum(axis=1)

print("\n(b) distribution of #positive classes per lesion:")
print(row_sums.value_counts())

violations_b = gt.loc[row_sums != 1, ["lesion_id"] + class_cols]
print("(b) lesions with != 1 positive class:", len(violations_b))
if len(violations_b):
    print(violations_b)

assert (row_sums == 1).all(), f"{len(violations_b)} lesions violate the one-hot assumption"
print("(b) PASSED: every lesion has exactly one positive class")


(b) distribution of #positive classes per lesion:
1    5240
Name: count, dtype: int64
(b) lesions with != 1 positive class: 0
(b) PASSED: every lesion has exactly one positive class


In [20]:
# ============================================================
# (c) does each class map to exactly one diagnosis_1?
# ============================================================
gt["class_name"] = gt[class_cols].idxmax(axis=1)  # collapse one-hot -> single label

lesion_diag1 = metadata.drop_duplicates("lesion_id")[["lesion_id", "diagnosis_1"]]
merged = gt[["lesion_id", "class_name"]].merge(lesion_diag1, on="lesion_id", how="inner")
assert len(merged) == len(gt), "merge dropped or duplicated rows — check for missing lesion_ids"

class_to_diag1 = (
    merged.groupby(["class_name", "diagnosis_1"])
    .size()
    .reset_index(name="lesion_count")
    .sort_values(["class_name", "diagnosis_1"])
)
print("\n(c) class -> diagnosis_1 table:")
print(class_to_diag1.to_string(index=False))

n_diag1_per_class = class_to_diag1.groupby("class_name")["diagnosis_1"].nunique()
inconsistent_classes = n_diag1_per_class[n_diag1_per_class > 1]

print("\n(c) classes mapping to MORE THAN ONE diagnosis_1:", list(inconsistent_classes.index))
if len(inconsistent_classes):
    print(class_to_diag1[class_to_diag1["class_name"].isin(inconsistent_classes.index)]
          .to_string(index=False))
else:
    print("(c) all 11 classes map to exactly one diagnosis_1")



(c) class -> diagnosis_1 table:
class_name   diagnosis_1  lesion_count
     AKIEC Indeterminate           123
     AKIEC     Malignant           180
       BCC     Malignant          2522
   BEN_OTH        Benign            44
       BKL        Benign           544
        DF        Benign            52
       INF        Benign            50
   MAL_OTH     Malignant             9
       MEL     Malignant           450
        NV        Benign           746
     SCCKA     Malignant           473
      VASC        Benign            47

(c) classes mapping to MORE THAN ONE diagnosis_1: ['AKIEC']
class_name   diagnosis_1  lesion_count
     AKIEC Indeterminate           123
     AKIEC     Malignant           180


In [21]:

# ============================================================
# (d) diagnosis hierarchy: diagnosis_3 -> diagnosis_2 -> diagnosis_1
# ============================================================
lesion_level = metadata.drop_duplicates("lesion_id")

def check_hierarchy(df, child, parent):
    sub = df[[child, parent]].dropna(subset=[child])
    counts = sub.groupby(child)[parent].nunique()
    bad = counts[counts > 1]
    print(f"\n(d) {child} -> {parent}: {len(bad)} inconsistent {child} value(s) "
          f"out of {len(counts)}")
    if len(bad):
        for val in bad.index:
            print(sub[sub[child] == val].drop_duplicates().to_string(index=False))
    return bad

bad_32 = check_hierarchy(lesion_level, "diagnosis_3", "diagnosis_2")
bad_21 = check_hierarchy(lesion_level, "diagnosis_2", "diagnosis_1")

assert bad_32.empty, "diagnosis_3 -> diagnosis_2 is not a clean hierarchy"
assert bad_21.empty, "diagnosis_2 -> diagnosis_1 is not a clean hierarchy"
print("(d) PASSED: diagnosis hierarchy is strictly nested at both levels")



(d) diagnosis_3 -> diagnosis_2: 0 inconsistent diagnosis_3 value(s) out of 27

(d) diagnosis_2 -> diagnosis_1: 0 inconsistent diagnosis_2 value(s) out of 19
(d) PASSED: diagnosis hierarchy is strictly nested at both levels


In [22]:
# ============================================================
# (e) the two images of a lesion agree on age, sex, site, diagnosis
# ============================================================
fields_to_check = ["age_approx", "sex", "anatom_site_general",
                    "diagnosis_1", "diagnosis_2", "diagnosis_3"]

print("\n(e) per-field consistency across each lesion's 2 images:")
for field in fields_to_check:
    nun = metadata.groupby("lesion_id")[field].nunique(dropna=False)
    bad_ids = nun[nun > 1].index.tolist()
    print(f"  {field}: {len(bad_ids)} lesions with inconsistent values")
    if bad_ids:
        print(metadata[metadata["lesion_id"].isin(bad_ids)]
              .sort_values("lesion_id")[["lesion_id", "isic_id", field]]
              .to_string(index=False))
    assert len(bad_ids) == 0, f"{field} disagrees within {len(bad_ids)} lesion(s)"

print("\n(e) PASSED: all checked fields agree between both images of every lesion")

print("\n" + "=" * 60)
print("ALL CHECKS COMPLETE")
print("=" * 60)


(e) per-field consistency across each lesion's 2 images:
  age_approx: 0 lesions with inconsistent values
  sex: 0 lesions with inconsistent values
  anatom_site_general: 0 lesions with inconsistent values
  diagnosis_1: 0 lesions with inconsistent values
  diagnosis_2: 0 lesions with inconsistent values
  diagnosis_3: 0 lesions with inconsistent values

(e) PASSED: all checked fields agree between both images of every lesion

ALL CHECKS COMPLETE


Check (c) shows that 10 of the 11 classes map to exactly one diagnosis_1 value (e.g. BCC → Malignant with 2,522 lesions, NV → Benign with 746), but AKIEC is the one exception: 180 of its lesions are labeled Malignant and 123 are labeled Indeterminate. This means the 11-class scheme is almost a strict refinement of diagnosis_1, but not fully — for any lesion predicted as one of the other 10 classes, diagnosis_1 can be recovered deterministically via a fixed lookup table, but for AKIEC predictions, diagnosis_1 is genuinely ambiguous and cannot be derived from the 11-class label alone. For the project's label strategy, this implies deriving diagnosis_1 from the 11-class prediction is safe for most of the dataset, but AKIEC would need either its own auxiliary classifier for diagnosis_1 or an explicitly documented error margin, since roughly 40% of AKIEC lesions (123 of 303) would be mis-derived by any single lookup rule. On any new medical dataset I would always repeat checks (a) and (b) before training, since they catch structural corruption — missing/orphaned ids and broken one-hot encoding — that doesn't crash the pipeline but silently injects wrong supervision; by contrast, (d) and (e) passing cleanly here confirms the diagnosis hierarchy itself and the per-lesion metadata are trustworthy, so those checks matter mainly when a dataset's hierarchy or multi-image structure hasn't already been validated. The most dangerous failure would be (b) going unnoticed: a lesion with zero positive classes trains as a phantom all-negative example, and one with multiple positive classes creates ambiguous multi-label gradient signal for what should be a single-label problem — both degrade the model quietly rather than raising an error, which is far worse than a loud failure like (a).

A1.2 Ask the data five questions.